In [1]:
import pandas as pd

df = pd.read_excel("Online Retail.xlsx")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [3]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [4]:
# Remove rows with missing CustomerID
df = df.dropna(subset=['CustomerID'])

# Remove negative or zero quantities (returns/cancellations)
df = df[df['Quantity'] > 0]

# Remove duplicates
df = df.drop_duplicates()

In [5]:
df['CustomerID'] = df['CustomerID'].astype(int)

In [6]:
df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]

In [8]:
customer = df.groupby('CustomerID').agg({
    'InvoiceNo': 'nunique',
    'TotalAmount': 'sum'
}).reset_index()

customer.columns = ['CustomerID', 'TotalOrders', 'TotalSpend']
customer

,CustomerID,TotalOrders,TotalSpend
0,12346,1,77183.60
1,12347,7,4310.00
2,12348,4,1797.24
3,12349,1,1757.55
4,12350,1,334.40
...,...,...,...
4334,18280,1,180.60
4335,18281,1,80.82
4336,18282,2,178.05
4337,18283,16,2045.53


In [9]:
customer['AOV'] = customer['TotalSpend'] / customer['TotalOrders']
customer['PurchaseFrequency'] = customer['TotalOrders']

In [ ]:
lifespan = 12
# Lifespan not available in dataset, so a fixed 12-month assumption is used for consistent CLTV estimation
customer['CLTV'] = customer['AOV'] * customer['PurchaseFrequency'] * lifespan

In [12]:
top_customers = customer.sort_values(by='CLTV', ascending=False).head(10)
top_customers

,CustomerID,TotalOrders,TotalSpend,AOV,PurchaseFrequency,CLTV
1690,14646,74,280206.02,3786.567838,74,3362472.24
4202,18102,60,259657.30,4327.621667,60,3115887.60
3729,17450,46,194390.79,4225.886739,46,2332689.48
3009,16446,2,168472.50,84236.250000,2,2021670.00
1880,14911,201,143711.17,714.980945,201,1724534.04
55,12415,21,124914.53,5948.310952,21,1498974.36
1334,14156,55,117210.08,2131.092364,55,1406520.96
3772,17511,31,91062.38,2937.496129,31,1092748.56
2703,16029,63,80850.84,1283.346667,63,970210.08
0,12346,1,77183.60,77183.600000,1,926203.20


# CLTV Calculation

Customer Lifetime Value (CLTV) was estimated using average order value and purchase frequency. A fixed customer lifespan of 12 months was assumed to project future revenue.

# Retention Strategy for High CLTV Customers

Top customers contribute highly to revenue and should be prioritized for retention. These customers can be retained through exclusive benefits such as early access to products, personalized offers based on purchase history, and priority customer support.

Additionally, incentivizing higher spending through bundled offers or premium product recommendations can further increase their lifetime value.

In [13]:
customer.to_csv("customer_cltv.csv", index=False)